[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch02/NN_DL_ch02_VaRForecasting/NN_DL_ch02_VaRForecasting.ipynb)

# NN_DL_ch02_VaRForecasting

**Value-at-Risk Forecasting with Neural Networks**

Download real S&P 500 data, compute daily returns, and train an MLP for 1-day VaR forecasting. Compare with historical simulation and evaluate with violation tests.

In [ ]:
!pip install -q torch matplotlib pandas scikit-learn

In [ ]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [ ]:
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load S&P 500 Data
import pandas as pd

try:
    url = 'https://raw.githubusercontent.com/datasets/s-and-p-500/main/data/data.csv'
    sp = pd.read_csv(url)
    # Find relevant columns
    date_cols = [c for c in sp.columns if 'date' in c.lower()]
    price_cols = [c for c in sp.columns if any(k in c.lower() for k in ['sp500', 'price', 'close', 'value'])]
    if not price_cols:
        price_cols = [c for c in sp.columns if c not in date_cols]
    sp = sp.dropna(subset=[price_cols[0]])
    prices = sp[price_cols[0]].values.astype(float)
    print(f"Loaded real S&P 500 data: {len(prices)} observations")
except Exception as e:
    print(f"Could not load remote data ({e}), generating realistic GARCH returns")
    np.random.seed(42)
    n = 5000
    omega, alpha, beta = 1e-6, 0.09, 0.90
    sigma2 = np.zeros(n)
    returns = np.zeros(n)
    sigma2[0] = omega / (1 - alpha - beta)
    for t in range(1, n):
        sigma2[t] = omega + alpha * returns[t-1]**2 + beta * sigma2[t-1]
        returns[t] = np.sqrt(sigma2[t]) * np.random.standard_t(df=5)
    prices = 100 * np.exp(np.cumsum(returns))
    print(f"Generated {n} GARCH(1,1) observations")

log_returns = np.diff(np.log(prices))
print(f"Returns: {len(log_returns)} observations")
print(f"Mean: {log_returns.mean():.6f}, Std: {log_returns.std():.4f}")
print(f"Min: {log_returns.min():.4f}, Max: {log_returns.max():.4f}")

In [ ]:
# Prepare Features for VaR Forecasting
window = 20
n = len(log_returns)

X_all, y_all = [], []
for t in range(window, n):
    past = log_returns[t-window:t]
    features = list(past) + [past.std(), past.mean(), np.min(past), np.max(past)]
    X_all.append(features)
    y_all.append(log_returns[t])

X_all = np.array(X_all, dtype=np.float32)
y_all = np.array(y_all, dtype=np.float32)

split = int(0.8 * len(X_all))
X_train, X_test = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]

from sklearn.preprocessing import StandardScaler
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# MLP for VaR (Quantile Regression)
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

alpha_var = 0.01

class VaR_MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1))
    def forward(self, x):
        return self.net(x)

def quantile_loss(pred, target, alpha):
    err = target - pred.squeeze()
    return torch.mean(torch.where(err >= 0, alpha * err, (alpha - 1) * err))

model = VaR_MLP(X_train.shape[1]).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

losses_hist = []
for epoch in range(200):
    model.train()
    ep_loss = []
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = quantile_loss(pred, yb, alpha_var)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        ep_loss.append(loss.item())
    losses_hist.append(np.mean(ep_loss))
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}: Quantile Loss = {losses_hist[-1]:.6f}")

# Predict VaR on test set
model.eval()
with torch.no_grad():
    var_nn = model(torch.FloatTensor(X_test).to(device)).cpu().numpy().ravel()

# Historical simulation VaR
var_hs = np.array([np.percentile(y_all[max(0,split+t-250):split+t], alpha_var*100)
                   for t in range(len(y_test))])

print(f"\nNN VaR violations: {(y_test < var_nn).mean():.4f} (target: {alpha_var})")
print(f"HS VaR violations: {(y_test < var_hs).mean():.4f} (target: {alpha_var})")

In [ ]:
# VaR Forecast Plot
fig, ax = plt.subplots(figsize=(16, 6))
t = np.arange(len(y_test))
ax.plot(t, y_test, color='grey', lw=0.5, alpha=0.6, label='Returns')
ax.plot(t, var_nn, color=MAIN_BLUE, lw=1.5, label=f'NN VaR ({(y_test < var_nn).mean():.3f})')
ax.plot(t, var_hs, color=IDA_RED,   lw=1.5, label=f'Hist. Sim. VaR ({(y_test < var_hs).mean():.3f})', ls='--')

violations_nn = y_test < var_nn
ax.scatter(t[violations_nn], y_test[violations_nn], color=CRIMSON, s=15, zorder=5, label='NN Violations')
ax.axhline(0, color='grey', lw=0.5)
ax.set_xlabel('Time'); ax.set_ylabel('Return')
ax.set_title(f'1% VaR Forecast: Neural Network vs Historical Simulation', fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
save_fig('nn_ch02_var_forecast')

In [ ]:
# VaR Violations Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

viol_nn = np.cumsum(y_test < var_nn)
viol_hs = np.cumsum(y_test < var_hs)
expected = np.arange(1, len(y_test)+1) * alpha_var

axes[0].plot(viol_nn, color=MAIN_BLUE, lw=2, label='NN')
axes[0].plot(viol_hs, color=IDA_RED, lw=2, label='Hist. Sim.', ls='--')
axes[0].plot(expected, color='grey', lw=1.5, ls=':', label='Expected')
axes[0].set_xlabel('Time'); axes[0].set_ylabel('Cumulative Violations')
axes[0].set_title('Cumulative VaR Violations', fontweight='bold'); axes[0].legend()

nn_viols = y_test[y_test < var_nn]
hs_viols = y_test[y_test < var_hs]
if len(nn_viols) > 0:
    axes[1].hist(nn_viols, bins=30, color=MAIN_BLUE, alpha=0.6, label='NN violations', edgecolor='white')
if len(hs_viols) > 0:
    axes[1].hist(hs_viols, bins=30, color=IDA_RED, alpha=0.4, label='HS violations', edgecolor='white')
axes[1].set_xlabel('Return'); axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of VaR Violations', fontweight='bold'); axes[1].legend()

plt.tight_layout()
save_fig('nn_ch02_var_violations')

In [ ]:
# Optimizer Comparison
optimizers_config = {
    'SGD': lambda p: optim.SGD(p, lr=0.01),
    'SGD+Momentum': lambda p: optim.SGD(p, lr=0.01, momentum=0.9),
    'Adam': lambda p: optim.Adam(p, lr=1e-3),
    'RMSprop': lambda p: optim.RMSprop(p, lr=1e-3),
}

fig, ax = plt.subplots(figsize=(12, 6))
colors_opt = [MAIN_BLUE, IDA_RED, FOREST, AMBER]

for (name, opt_fn), color in zip(optimizers_config.items(), colors_opt):
    torch.manual_seed(42)
    m = VaR_MLP(X_train.shape[1]).to(device)
    opt = opt_fn(m.parameters())
    hist = []
    for epoch in range(100):
        m.train()
        ep_loss = []
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = m(xb)
            loss = quantile_loss(pred, yb, alpha_var)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss.append(loss.item())
        hist.append(np.mean(ep_loss))
    ax.plot(hist, color=color, lw=2, label=name)

ax.set_xlabel('Epoch'); ax.set_ylabel('Quantile Loss')
ax.set_title('Optimizer Comparison for VaR Forecasting', fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
save_fig('nn_ch02_optimizer_comparison')

## Summary

Trained a quantile-regression MLP for **1% Value-at-Risk** forecasting on S&P 500 returns. Compared with historical simulation and evaluated via violation rates. Also compared SGD, Adam, and RMSprop convergence.